In [ ]:
!pip install roboflow ultralytics

In [ ]:
!pip install tensorflow==2.21.0 keras==3.14.1

In [ ]:
from google.colab import drive
from google.colab import userdata

import json
import numpy as np
import tensorflow
from tensorflow.keras.layers import Flatten, Dense, ReLU, Dense, Activation
from pathlib import Path

In [ ]:
drive.mount('/content/drive')

project_directory_path = "/content/drive/MyDrive/Colab Notebooks/Projects/PAAI Project"
additional_dataset_directory_path = project_directory_path + "/additional_dataset"

dataset_cropped_directory_path = project_directory_path + "/dataset_cropped"
cropped_json = dataset_cropped_directory_path + "/project-2-at-2026-06-07-19-37-c29d30b5.json"

---

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
VAL_SPLIT  = 0.2

In [ ]:
def parse_ls_export(json_path, images_dir):
    with open(json_path) as f:
        tasks = json.load(f)

    image_paths, labels = [], []

    for task in tasks:
        filename = Path(task["image"]).name
        path = str(Path(images_dir) / filename)

        points = {}
        for kp in task["keypoints"]:
            label = kp["keypointlabels"][0]
            points[label] = (kp["x"] / 100.0, kp["y"] / 100.0)

        try:
            row = [
                points["tl"][0], points["tl"][1],
                points["tr"][0], points["tr"][1],
                points["br"][0], points["br"][1],
                points["bl"][0], points["bl"][1],
            ]
        except KeyError:
            continue

        image_paths.append(path)
        labels.append(row)

    return image_paths, np.array(labels, dtype=np.float32)

In [ ]:
image_paths, labels = parse_ls_export(cropped_json, dataset_cropped_directory_path)
print(f"{len(image_paths)} labeled images")

In [ ]:
def load_and_preprocess(path, label):
    img = tensorflow.io.read_file(path)
    img = tensorflow.image.decode_jpeg(img, channels=3)
    img = tensorflow.image.resize(img, IMG_SIZE)
    return img, label